# Arrival delays on flights from RDU

Which destinations had the worst average arrival delays from Raleigh–Durham between 2004 and 2008? Did the share of late arrivals change over those five years?

The input is ten annual CSVs (1999–2008) from the [ASA Data Expo airline on-time dataset](https://doi.org/10.7910/DVN/HG7NV7), available in Databricks. The earlier years also give the partition filter something to skip when we run the five-year analysis.

Connect to **Serverless** and run the cells in order. The main pipeline writes two Delta tables. The final two sections use the filtered flights again to benchmark caching and try a simple delay prediction model.

## 1. Check the input files

Before running Spark, check that all ten files exist and add up to at least 1 GB. The size comes from file metadata, so this check avoids a full scan just to prove the dataset is large enough.

In [0]:
from time import perf_counter

from pyspark.sql import functions as F


SOURCE_DIR = "/databricks-datasets/asa/airlines"
SOURCE_YEARS = range(1999, 2009)
MIN_SOURCE_BYTES = 1_000_000_000
FIRST_ANALYSIS_YEAR = 2004
LAST_ANALYSIS_YEAR = 2008
ORIGIN = "RDU"

# Change the catalog/schema here if your workspace uses a different writable schema.
STAGING_TABLE = "workspace.default.ids706_flight_stage"
RESULT_TABLE = "workspace.default.ids706_rdu_route_delay"

# Match the exact years so an extra file cannot change the analysis input.
files_by_name = {entry.name: entry for entry in dbutils.fs.ls(SOURCE_DIR)}
expected_names = [f"{year}.csv" for year in SOURCE_YEARS]
missing_names = [name for name in expected_names if name not in files_by_name]
if missing_names:
    raise FileNotFoundError(f"Missing annual CSV files: {', '.join(missing_names)}")

source_files = [files_by_name[name] for name in expected_names]
source_bytes = sum(entry.size for entry in source_files)
print(f"Source: {len(source_files)} CSV files, {source_bytes:,} bytes "
      f"({source_bytes / 1e9:.3f} GB)")
if source_bytes < MIN_SOURCE_BYTES:
    raise ValueError("The selected source files must total at least 1 GB.")

Source: 10 CSV files, 6,286,007,363 bytes (6.286 GB)


## 2. Read the CSVs and write a Delta staging table

Keep the five columns the analysis uses and convert the numeric fields from text. Staging all ten years by `Year` lets the next query read just the years it needs. This write is the first big job in the notebook.

In [0]:
raw = spark.read.option("header", True).csv([entry.path for entry in source_files])
# try_cast turns missing or malformed numbers such as NA into nulls.
typed_flights = (
    raw.select("Year", "Origin", "Dest", "Cancelled", "ArrDelay")
    .withColumn("Year", F.expr("try_cast(Year AS INT)"))
    .withColumn("Cancelled", F.expr("try_cast(Cancelled AS INT)"))
    .withColumn("ArrDelay", F.expr("try_cast(ArrDelay AS INT)"))
)

stage_start = perf_counter()
(
    typed_flights.write.format("delta")
    .mode("overwrite")
    .partitionBy("Year")  # Store each year separately for partition pruning.
    .saveAsTable(STAGING_TABLE)
)
stage_seconds = perf_counter() - stage_start
print(f"Saved year-partitioned Delta staging table: {STAGING_TABLE} "
      f"({stage_seconds:.2f} seconds)")

Saved year-partitioned Delta staging table: workspace.default.ids706_flight_stage (25.94 seconds)


## 3. Define the route summary

Start with noncancelled flights leaving RDU that have a destination and a recorded arrival delay. A flight counts as late if it arrived **more than 15 minutes** after schedule. Negative delays are early arrivals and stay in the average.

For each year and destination, calculate flight count, mean delay, the approximate 90th percentile, and the share of late flights. Keep the delay and late-flight totals so the SQL queries can combine years or destinations with the right weights.

This cell builds the query. Its printed time measures plan construction in Python. Spark computes the route totals when we write the results in Section 5.

In [0]:
# Transformation demo: construct the query before any action on it.
build_start = perf_counter()
stage = spark.table(STAGING_TABLE)
eligible_flights = (
    stage
    .filter(F.col("Year").between(FIRST_ANALYSIS_YEAR, LAST_ANALYSIS_YEAR))
    .filter(F.col("Origin") == ORIGIN)
    .filter(F.col("Cancelled") == 0)
    .filter(F.col("Dest").isNotNull() & (F.col("Dest") != ""))
    .filter(F.col("ArrDelay").isNotNull())
    .select("Year", "Dest", "ArrDelay")
    .withColumn("is_late", (F.col("ArrDelay") > 15).cast("int"))
)
# One output row per year and destination, with totals for the later SQL.
route_delay = (
    eligible_flights.groupBy("Year", "Dest")
    .agg(
        F.count("*").alias("flights"),
        F.sum("ArrDelay").alias("sum_arrival_delay_minutes"),
        F.avg("ArrDelay").alias("avg_arrival_delay_minutes"),
        F.sum("is_late").alias("late_flights"),
        # The 90th percentile shows what the upper end of the delays looks like.
        F.percentile_approx("ArrDelay", 0.90, 1000).alias("p90_arrival_delay_minutes"),
    )
    .withColumn("late_pct", 100.0 * F.col("late_flights") / F.col("flights"))
)
build_seconds = perf_counter() - build_start
print(f"Constructed the route query in {build_seconds:.4f} seconds. "
      "The route aggregation has not run yet.")

Constructed the route query in 0.0024 seconds. The route aggregation has not run yet.


## 4. Inspect the physical plan

Look at the scan for the year partition filter, the columns Spark reads, and the other predicates it can apply there. Then look for a partial aggregate before the exchange. Local aggregation can reduce what Spark sends across workers. Use the actual operator names and predicates shown in your output for the README.

In [0]:
# Capture the scan, filter, partial aggregation and exchange in this cell.
route_delay.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonGroupingAgg (7)
            +- PhotonShuffleExchangeSource (6)
               +- PhotonShuffleMapStage (5)
                  +- PhotonShuffleExchangeSink (4)
                     +- PhotonGroupingAgg (3)
                        +- PhotonProject (2)
                           +- PhotonScan parquet workspace.default.ids706_flight_stage (1)


(1) PhotonScan parquet workspace.default.ids706_flight_stage
Output [5]: [Origin#11985, Dest#11986, Cancelled#11987, ArrDelay#11988, Year#11984]
DictionaryFilters: [NOT (Dest#11986 = ), (Cancelled#11987 = 0), (Origin#11985 = RDU)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-hjfpv38372/uc/e37f889f-6e80-4ee1-860b-a825318e4041/c8e3cf42-e189-44c1-9a48-b61512555bb3/__unitystorage/catalogs/6715a7e7-39f5-4c30-9b5f-bad664b61bbd/tables/e8f90347-1dbb-4078-bda8-91e9ef08cc5a]
PartitionFilt

## 5. Run the query and save the results

`saveAsTable()` triggers the route query and writes the small summary as a Delta table. After the write finishes, open **See performance** for this cell and select the write statement to capture the Query Details or Query Profile. The operator metrics here help identify where the query spent its time.

In [0]:
# Action demo: the write triggers the scan and aggregation and persists results.
action_start = perf_counter()
route_delay.write.format("delta").mode("overwrite").saveAsTable(RESULT_TABLE)
action_seconds = perf_counter() - action_start
print(f"Saved Delta results: {RESULT_TABLE} ({action_seconds:.2f} seconds)")

Saved Delta results: workspace.default.ids706_rdu_route_delay (2.52 seconds)


## 6. Read back the saved summary

Check that the saved table has results for all five years. The preview shows eight rows so you can see what a route-year record looks like.

In [0]:
# Read back the persisted results. This is a separate action on the small table.
summary = spark.table(RESULT_TABLE).agg(
    F.count("*").alias("route_year_rows"),
    F.sum("flights").alias("eligible_flights"),
    F.countDistinct("Year").alias("years"),
).first()
if (
    summary is None
    or summary['route_year_rows'] == 0
    or summary['years'] != LAST_ANALYSIS_YEAR - FIRST_ANALYSIS_YEAR + 1
):
    raise RuntimeError("The result table is empty or does not cover all analysis years.")

print(f"Pipeline successful: {summary['route_year_rows']:,} route-year rows, "
      f"{summary['eligible_flights']:,} eligible flights, {summary['years']} years.")
spark.table(RESULT_TABLE).orderBy("Year", "Dest").show(8, truncate=False)

Pipeline successful: 180 route-year rows, 302,707 eligible flights, 5 years.
+----+----+-------+-------------------------+-------------------------+------------+-------------------------+------------------+
|Year|Dest|flights|sum_arrival_delay_minutes|avg_arrival_delay_minutes|late_flights|p90_arrival_delay_minutes|late_pct          |
+----+----+-------+-------------------------+-------------------------+------------+-------------------------+------------------+
|2004|ATL |5497   |59715                    |10.863198108058942       |1373        |44                       |24.977260323812988|
|2004|AUS |176    |1078                     |6.125                    |39          |50                       |22.15909090909091 |
|2004|BDL |1202   |12144                    |10.103161397670549       |285         |51                       |23.710482529118135|
|2004|BNA |1711   |11846                    |6.92343658679135         |304         |33                       |17.76738749269433 |
|2004|BOS |38

## 7. Find the five destinations with the highest average delay

Require at least 1,000 eligible flights across the five years. Divide the total delay by the total number of flights so a year with 5,000 flights contributes more than a year with 500.

In [0]:
# SQL 1: rank destinations over all five years, keeping at least 1,000 flights.
# Use underlying totals so each flight gets equal weight.
worst_routes = spark.sql(f"""
    WITH destination_totals AS (
        SELECT Dest, SUM(flights) AS flights,
               SUM(sum_arrival_delay_minutes) AS delay_minutes,
               SUM(late_flights) AS late_flights
        FROM {RESULT_TABLE}
        GROUP BY Dest
        HAVING SUM(flights) >= 1000
    )
    SELECT Dest, flights,
           ROUND(delay_minutes / flights, 2) AS avg_delay_minutes,
           ROUND(100.0 * late_flights / flights, 2) AS late_pct
    FROM destination_totals
    ORDER BY delay_minutes / flights DESC, Dest
    LIMIT 5
""")
worst_routes.show(truncate=False)

+----+-------+-----------------+--------+
|Dest|flights|avg_delay_minutes|late_pct|
+----+-------+-----------------+--------+
|EWR |18235  |21.14            |34.32   |
|LGA |22429  |15.34            |32.70   |
|AUS |2195   |14.83            |32.76   |
|ORD |13588  |13.67            |25.77   |
|JFK |16343  |13.31            |27.15   |
+----+-------+-----------------+--------+



## 8. Compare the five years

Combine all destinations within each year. The late percentage uses total late flights divided by total eligible flights, which gives each flight equal weight.

In [0]:
# SQL 2: compare annual delay rates and average delay across all destinations.
annual_delays = spark.sql(f"""
    SELECT Year, SUM(flights) AS flights,
           ROUND(SUM(sum_arrival_delay_minutes) / SUM(flights), 2) AS avg_delay_minutes,
           ROUND(100.0 * SUM(late_flights) / SUM(flights), 2) AS late_pct
    FROM {RESULT_TABLE}
    GROUP BY Year
    ORDER BY Year
""")
annual_delays.show(truncate=False)

+----+-------+-----------------+--------+
|Year|flights|avg_delay_minutes|late_pct|
+----+-------+-----------------+--------+
|2004|60417  |6.29             |19.93   |
|2005|61259  |7.08             |20.05   |
|2006|56672  |9.96             |23.33   |
|2007|65569  |9.36             |22.79   |
|2008|58790  |8.15             |21.42   |
+----+-------+-----------------+--------+



## 9. Measure the effect of caching

Run the same delay summary with and without caching the filtered inputs. We warm up once, time three runs for each condition, and show the median. The first cached run fills the cache, so its time is reported separately. A speedup near 1× means this workload saw little difference in this session.

Databricks Serverless requires environment version 5 or later and the DataFrame caching feature enabled for the workspace.

In [0]:
from statistics import median

# These are the only two columns the repeated summary needs.
delay_input = eligible_flights.select("ArrDelay", "is_late")


def timed_delay_summary(df):
    start = perf_counter()
    # first() triggers the aggregation and brings back just one row.
    totals = df.agg(
        F.count("*").alias("flights"),
        F.sum("ArrDelay").alias("delay_minutes"),
        F.sum("is_late").alias("late_flights"),
    ).first()
    return perf_counter() - start, totals


# Give the session one untimed run before comparing repeated queries.
timed_delay_summary(delay_input)
uncached_runs = [timed_delay_summary(delay_input) for _ in range(3)]

cached_input = delay_input.cache()
try:
    # The first action after cache() materializes the cached rows.
    fill_seconds, filled_totals = timed_delay_summary(cached_input)
    cached_runs = [timed_delay_summary(cached_input) for _ in range(3)]
finally:
    cached_input.unpersist()

# All runs should return the same integer totals.
if any(totals != filled_totals for _, totals in uncached_runs + cached_runs):
    raise RuntimeError("The repeated summaries returned different totals.")

uncached_median = median(seconds for seconds, _ in uncached_runs)
cached_median = median(seconds for seconds, _ in cached_runs)

print(f"Without cache, median of 3 runs: {uncached_median:.3f} s")
print(f"First run with cache (includes filling it): {fill_seconds:.3f} s")
print(f"With cache already filled, median of 3 runs: {cached_median:.3f} s")
print(f"Uncached / cached runtime: {uncached_median / cached_median:.2f}x")

Without cache, median of 3 runs: 0.670 s
First run with cache (includes filling it): 0.816 s
With cache already filled, median of 3 runs: 0.657 s
Uncached / cached runtime: 1.02x


## 10. Try predicting delays with Spark MLlib

How much can we tell about a flight's chance of arriving late from its destination alone? Train logistic regression on 2004–2007 and test on 2008. Destination is known before departure. The actual arrival delay is used only to create the label.

Report the model's accuracy alongside a baseline that predicts every flight will arrive no more than 15 minutes late. Also report ROC AUC, which shows how well the scores rank late flights above the rest. This is a small experiment on flights with a recorded arrival delay.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

# Year sets the split. The only feature going into the model is Dest.
model_input = eligible_flights.select(
    "Year", "Dest", F.col("is_late").cast("double").alias("label")
)
test_year = LAST_ANALYSIS_YEAR
train = model_input.filter(F.col("Year") < test_year)
test = model_input.filter(F.col("Year") == test_year)

# Keep a bucket for destinations that first appear in the test year.
dest_indexer = StringIndexer(
    inputCol="Dest", outputCol="dest_index", handleInvalid="keep"
)
# Airport codes are categories. Give each one an indicator column in the vector.
dest_encoder = OneHotEncoder(
    inputCols=["dest_index"], outputCols=["dest_encoded"], dropLast=False
)
assembler = VectorAssembler(inputCols=["dest_encoded"], outputCol="features")
classifier = LogisticRegression(maxIter=30, regParam=0.01)

# Fit the destination encoding and the classifier using training data only.
model = Pipeline(stages=[dest_indexer, dest_encoder, assembler, classifier]).fit(train)
predictions = model.transform(test)

# A high accuracy can come from predicting the majority class every time.
metrics = predictions.agg(
    F.count("*").alias("test_flights"),
    F.avg("label").alias("late_rate"),
    F.avg((F.col("prediction") == F.col("label")).cast("double")).alias("accuracy"),
).first()

roc_auc = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(predictions)

print(f"Training years: {FIRST_ANALYSIS_YEAR}–{test_year - 1}")
print(f"Test set: {test_year}, {metrics['test_flights']:,} flights")
print(f"Late flights in the test set: {metrics['late_rate']:.1%}")
print(f"Baseline accuracy (predict no late arrivals): {1 - metrics['late_rate']:.1%}")
print(f"Logistic regression accuracy: {metrics['accuracy']:.1%}")
print(f"ROC AUC: {roc_auc:.3f}")

Training years: 2004–2007
Test set: 2008, 58,790 flights
Late flights in the test set: 21.4%
Baseline accuracy (predict no late arrivals): 78.6%
Logistic regression accuracy: 78.6%
ROC AUC: 0.579
